# Vega-Altair: tutorial warsztatowy

## Plan

| Segment | Czas | Co robimy |
|---|---:|---|
| Wprowadzenie | 5 min | Czym jest Altair, kiedy warto go uzyc, gdzie sprawdza sie w praktyce |
| Instalacja / srodowisko | 3-5 min | Instalacja, Jupyter, wymagania, pierwsze uruchomienie |
| Praca na realnym zbiorze danych | 20 min | Import, transformacja, 3 wizualizacje i interakcja |
| Funkcje zaawansowane | 5-8 min | Duze dane, customizacja, wydajnosc |
| Mini-zadanie i dyskusja | 5-8 min | Krotki challenge dla grupy i pytania |


## 1. Wprowadzenie (5 min)

### Czym jest narzedzie?

Vega-Altair to biblioteka Pythona do tworzenia wykresow w sposob **deklaratywny**. Zamiast opisywac krok po kroku, jak narysowac linie, punkty albo osie, opisujemy:

- jakie dane chcemy pokazac,
- ktore kolumny mapujemy na os X, os Y, kolor, rozmiar, tooltip,
- jakiego typu znaku graficznego uzywamy: punktow, linii, slupkow, prostokatow,
- jakie transformacje i interakcje maja dzialac na wykresie.

Altair generuje specyfikacje Vega-Lite, ktora moze byc renderowana w notebookach, stronach WWW i dashboardach.

### Do jakich problemow sie nadaje?

Altair jest szczegolnie dobry, gdy chcemy szybko i czytelnie eksplorowac dane tabelaryczne:

- analiza eksploracyjna danych w notebooku,
- porownywanie rozkladow i trendow,
- szybkie prototypowanie dashboardow,
- wizualizacje interaktywne bez recznego pisania JavaScriptu,
- pokazywanie zaleznosci miedzy kilkoma zmiennymi.

Mniej pasuje do bardzo niestandardowych, recznie rysowanych infografik albo aplikacji, w ktorych potrzebna jest pelna kontrola nad kazdym pikselem.

### Przyklady zastosowan w przemysle

- **Finanse:** porownanie ryzyka i zwrotu portfeli, trendy cen, anomalie transakcyjne.
- **E-commerce:** analiza lejka zakupowego, segmentacja klientow, sezonowosc sprzedazy.
- **Produkcja:** monitoring parametrow maszyn, wykrywanie odchylen w czasie.
- **Medycyna / health-tech:** eksploracja wynikow badan, porownywanie kohort pacjentow.
- **Data science / ML:** analiza cech, bledow modelu, zaleznosci miedzy predykcja a rzeczywistoscia.


## 2. Instalacja / srodowisko (3-5 min)

### Jak zaczac

Najwygodniej pracowac w Jupyter Notebook, JupyterLab, VS Code z obsluga notebookow albo Google Colab.

Oficjalna dokumentacja Vega-Altair 6 zaleca instalacje pelnego zestawu zaleznosci przez:

```bash
pip install "altair[all]"
```

W tym tutorialu uzywamy dodatkowo `pandas`, `vega_datasets` i opcjonalnie `vegafusion` do wiekszych danych.

### Wymagania

- Python 3.10+ bedzie bezpiecznym wyborem dla aktualnych wersji bibliotek.
- Notebook albo IDE potrafiace wyswietlac HTML/JavaScript.
- Podstawowa znajomosc `pandas`: `read_csv`, kolumny, grupowanie, filtrowanie.
- Polaczenie z internetem przy pierwszym pobraniu danych z `vega_datasets`.


In [ ]:
!pip install "altair[all]" pandas vega_datasets vegafusion

In [39]:
import altair as alt
import pandas as pd
from vega_datasets import data

print("Altair:", alt.__version__)
print("Pandas:", pd.__version__)

Altair: 6.1.0
Pandas: 3.0.2


## 3. Praca na realnym zbiorze danych (20 min)

Uzyjemy zbioru **Seattle Weather** z repozytorium `vega_datasets`. To dzienny zbior obserwacji pogodowych z kolumnami m.in.:

- `date` - data pomiaru,
- `precipitation` - opady,
- `temp_max`, `temp_min` - temperatura maksymalna i minimalna,
- `wind` - predkosc wiatru,
- `weather` - kategoria pogody.

Bedziemy chcieli odpowiedziec na pytania:

- jak zmienia sie temperatura w czasie,
- ktore miesiace maja najwieksze opady,
- czy widac zaleznosc miedzy temperatura, wiatrem i typem pogody,
- jak interaktywnie filtrowac dane na wykresie.


### 3.1 Import danych

Altair moze przyjmowac dane jako `pandas.DataFrame` albo jako URL do pliku z danymi

In [40]:
weather_url = data.seattle_weather.url
weather = pd.read_csv(weather_url, parse_dates=["date"])

weather.head()

,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,2012-01-02,10.9,10.6,2.8,4.5,rain
2,2012-01-03,0.8,11.7,7.2,2.3,rain
3,2012-01-04,20.3,12.2,5.6,4.7,rain
4,2012-01-05,1.3,8.9,2.8,6.1,rain


In [41]:
weather.info()
weather.describe(include="all")

<class 'pandas.DataFrame'>
RangeIndex: 1461 entries, 0 to 1460
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           1461 non-null   datetime64[us]
 1   precipitation  1461 non-null   float64       
 2   temp_max       1461 non-null   float64       
 3   temp_min       1461 non-null   float64       
 4   wind           1461 non-null   float64       
 5   weather        1461 non-null   str           
dtypes: datetime64[us](1), float64(4), str(1)
memory usage: 73.4 KB


,date,precipitation,temp_max,temp_min,wind,weather
count,1461,1461.000000,1461.000000,1461.000000,1461.000000,1461
unique,NaN,NaN,NaN,NaN,NaN,5
top,NaN,NaN,NaN,NaN,NaN,sun
freq,NaN,NaN,NaN,NaN,NaN,714
mean,2013-12-31 00:00:00,3.029432,16.439083,8.234771,3.241136,NaN
min,2012-01-01 00:00:00,0.000000,-1.600000,-7.100000,0.400000,NaN
25%,2012-12-31 00:00:00,0.000000,10.600000,4.400000,2.200000,NaN
50%,2013-12-31 00:00:00,0.000000,15.600000,8.300000,3.000000,NaN
75%,2014-12-31 00:00:00,2.800000,22.200000,12.200000,4.000000,NaN
max,2015-12-31 00:00:00,55.900000,35.600000,18.300000,9.500000,NaN


### 3.2 Transformacja danych

Dodamy kilka kolumn pomocniczych:

- `year` i `month` do agregacji czasowej,
- `month_name` do czytelnych etykiet,
- `temp_mid` jako srednia z temperatury minimalnej i maksymalnej,
- `rain_day` jako flaga dnia z opadem.


In [42]:
weather_clean = weather.copy()
weather_clean["year"] = weather_clean["date"].dt.year
weather_clean["month"] = weather_clean["date"].dt.month
weather_clean["month_name"] = weather_clean["date"].dt.month_name()
weather_clean["temp_mid"] = (weather_clean["temp_min"] + weather_clean["temp_max"]) / 2
weather_clean["rain_day"] = weather_clean["precipitation"] > 0

weather_clean.head()

,date,precipitation,temp_max,temp_min,wind,weather,year,month,month_name,temp_mid,rain_day
0,2012-01-01,0.0,12.8,5.0,4.7,drizzle,2012,1,January,8.90,False
1,2012-01-02,10.9,10.6,2.8,4.5,rain,2012,1,January,6.70,True
2,2012-01-03,0.8,11.7,7.2,2.3,rain,2012,1,January,9.45,True
3,2012-01-04,20.3,12.2,5.6,4.7,rain,2012,1,January,8.90,True
4,2012-01-05,1.3,8.9,2.8,6.1,rain,2012,1,January,5.85,True


In [43]:
monthly = (
    weather_clean
    .groupby(["year", "month", "month_name"], as_index=False)
    .agg(
        avg_temp=("temp_mid", "mean"),
        total_precipitation=("precipitation", "sum"),
        rainy_days=("rain_day", "sum"),
        avg_wind=("wind", "mean"),
    )
)

monthly.head()

,year,month,month_name,avg_temp,total_precipitation,rainy_days,avg_wind
0,2012,1,January,4.298387,173.3,22,3.900000
1,2012,2,February,6.239655,92.3,19,3.903448
2,2012,3,March,6.196774,183.0,22,4.248387
3,2012,4,April,10.433333,68.1,17,3.373333
4,2012,5,May,12.925806,52.2,10,3.354839


### 3.3 Wizualizacji

Przykład wizualizacji:

```python
alt.Chart(dane).mark_...(...).encode(...)
```

Czyli:

- `alt.Chart(dane)` - z jakich danych korzystamy,
- `mark_...(...)` - jaka forma graficzna ma pokazac dane i jakie ma stale opcje wygladu,
- `encode(...)` - ktore kolumny danych trafiaja na osie, kolor, rozmiar, tooltip itd.

#### Co wybiera `mark_*()`?

`mark_*()` wybiera znak graficzny, czyli podstawowy element rysowany na wykresie. Ten sam zbior danych mozna pokazac jako slupki, linie, punkty, heatmape albo mape - zmienia sie glownie wlasnie `mark_*()`.

Najczestsze warianty:

| Metoda | Do czego sluzy | Typowy przyklad |
|---|---|---|
| `mark_bar()` | wykres slupkowy | liczba rekordow w kategoriach, suma sprzedazy |
| `mark_line()` | wykres liniowy | trend w czasie |
| `mark_point()` | punkty | scatter plot, relacja X-Y |
| `mark_circle()` | punkty jako kola | scatter plot z rozmiarem i kolorem |
| `mark_square()` | punkty jako kwadraty | alternatywa dla punktow/kolek |
| `mark_rect()` | prostokaty | heatmapa, macierz, kalendarz |
| `mark_area()` | wykres obszarowy | trend z wypelnieniem, stacked area |
| `mark_tick()` | male kreski | rozklad jednej zmiennej |
| `mark_rule()` | linia referencyjna | srednia, prog, granica |
| `mark_text()` | tekst na wykresie | etykiety wartosci |
| `mark_boxplot()` | boxplot | rozklad i odstajace wartosci |
| `mark_errorbar()` | przedzial bledu | srednia z niepewnoscia |
| `mark_geoshape()` | geometria mapy | mapa kraju, stanow, hrabstw |

#### Opcje przekazywane do `mark_*()`

Do `mark_*()` mozemy przekazac opcje, ktore ustawiaja **staly wyglad** znaku na calym wykresie. Na przyklad `mark_circle(size=80, opacity=0.7)` oznacza, ze wszystkie punkty maja bazowy rozmiar 80 i przezroczystosc 0.7.

```python
alt.Chart(df).mark_bar(color="#2563eb", cornerRadius=3)
alt.Chart(df).mark_line(point=True, strokeWidth=3)
alt.Chart(df).mark_circle(size=80, opacity=0.7, filled=True)
```

| Opcja | Co robi | Przyklad |
|---|---|---|
| `color` | staly kolor znaku | `mark_bar(color="#2563eb")` |
| `opacity` | przezroczystosc | `mark_circle(opacity=0.6)` |
| `size` | rozmiar punktu/tekstu | `mark_circle(size=80)` |
| `point` | dodaje punkty do linii | `mark_line(point=True)` |
| `line` | dodaje kontur do area | `mark_area(line=True)` |
| `stroke` | kolor obrysu | `mark_geoshape(stroke="white")` |
| `strokeWidth` | grubosc obrysu | `mark_line(strokeWidth=3)` |
| `cornerRadius` | zaokraglenie slupkow/prostokatow | `mark_bar(cornerRadius=3)` |
| `filled` | czy punkt ma byc wypelniony | `mark_point(filled=True)` |
| `tooltip` | wlacza prosty tooltip | `mark_point(tooltip=True)` |

#### `mark_*()` czy `encode()`?

Najprostsza zasada: jesli cos ma byc **takie samo dla wszystkich znakow**, dajemy to w `mark_*()`. Jesli cos ma **zalezec od danych**, dajemy to w `encode()`.

```python
# staly kolor wszystkich slupkow
alt.Chart(df).mark_bar(color="#2563eb").encode(x="category:N", y="value:Q")

# kolor zalezy od kolumny category
alt.Chart(df).mark_bar().encode(x="category:N", y="value:Q", color="category:N")
```

#### Co robi `encode()`?

`encode()` mowi Altairowi, **ktore kolumny danych maja sterowac ktorymi elementami wizualnymi**. To jest mapowanie danych na kanaly wizualne.

Najczestsze kanaly w `encode()`:

| Kanal | Co kontroluje | Przyklad |
|---|---|---|
| `x` | polozenie na osi X | `x="date:T"` |
| `y` | polozenie na osi Y | `y="sales:Q"` |
| `color` | kolor zalezny od danych | `color="weather:N"` |
| `size` | rozmiar punktu / znaku | `size="population:Q"` |
| `shape` | ksztalt punktu | `shape="species:N"` |
| `opacity` | przezroczystosc zalezna od danych | `opacity="score:Q"` |
| `tooltip` | dane widoczne po najechaniu | `tooltip=["date:T", "value:Q"]` |
| `facet` | podzial na male wykresy | `facet="year:O"` |
| `row` / `column` | uklad malych wykresow w siatce | `column="region:N"` |
| `order` | kolejnosc rysowania lub laczenia punktow | `order="date:T"` |

W `encode()` czesto pojawia sie zapis `"kolumna:typ"`. To nie jest dodatkowa zmienna, tylko informacja dla Altaira, jak traktowac dana kolumne.

| Typ | Pelna nazwa | Kiedy uzywac | Przyklad |
|---|---|---|---|
| `:Q` | quantitative | liczby ciagle | `"avg_temp:Q"`, `"price:Q"` |
| `:N` | nominal | kategorie bez porzadku | `"weather:N"`, `"city:N"` |
| `:O` | ordinal | kategorie z porzadkiem | `"month_name:O"`, `"rating:O"` |
| `:T` | temporal | daty i czas | `"date:T"`, `"yearmonth:T"` |

Przyklad z naszego wykresu:

```python
x=alt.X("yearmonth:T", title="Miesiac")
y=alt.Y("avg_temp:Q", title="Srednia temperatura")
color=alt.Color("weather:N", title="Pogoda")
tooltip=["date:T", "avg_temp:Q", "weather:N"]
```

Czyli `yearmonth` trafia na os X jako data, `avg_temp` trafia na os Y jako liczba, `weather` moze sterowac kolorem jako kategoria, a `tooltip` pokazuje dodatkowe informacje po najechaniu kursorem.

W naszym pierwszym wykresie uzywamy `mark_line(point=True)`, bo chcemy pokazac trend w czasie i jednoczesnie zaznaczyc konkretne miesieczne obserwacje punktami.


### 3.4 Wizualizacja 1: trend temperatury w czasie

Pierwszy wykres pokazuje srednia miesieczna temperature.

In [44]:
temp_line = (
    alt.Chart(monthly)
    .mark_line(point=True)
    .encode(
        x=alt.X("yearmonth:T", title="Miesiac"),
        y=alt.Y("avg_temp:Q", title="Srednia temperatura"),
        tooltip=[
            alt.Tooltip("year:O", title="Rok"),
            alt.Tooltip("month_name:N", title="Miesiac"),
            alt.Tooltip("avg_temp:Q", title="Srednia temp.", format=".1f"),
        ],
    )
    .transform_calculate(yearmonth="datetime(datum.year, datum.month - 1, 1)")
    .properties(width=750, height=300, title="Srednia miesieczna temperatura w Seattle")
)

temp_line

alt.Chart(...)

In [ ]:
# Vega
vega_json = temp_line.to_json(format="vega")

print(vega_json)

In [ ]:
# Vega lite
vl_json = temp_line.to_json()

print(vl_json)

### 3.5 Wizualizacja 2: opady wedlug miesiaca i roku

Heatmapa dobrze pokazuje sezonowosc. Kolor koduje laczna sume opadow w danym miesiacu.

In [46]:
month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]

precip_heatmap = (
    alt.Chart(monthly)
    .mark_rect()
    .encode(
        x=alt.X("year:O", title="Rok"),
        y=alt.Y("month_name:N", title="Miesiac", sort=month_order),
        color=alt.Color(
            "total_precipitation:Q",
            title="Suma opadow",
            scale=alt.Scale(scheme="tealblues"),
        ),
        tooltip=[
            alt.Tooltip("year:O", title="Rok"),
            alt.Tooltip("month_name:N", title="Miesiac"),
            alt.Tooltip("total_precipitation:Q", title="Suma opadow", format=".1f"),
            alt.Tooltip("rainy_days:Q", title="Dni z opadem"),
        ],
    )
    .properties(width=520, height=320, title="Opady w Seattle: sezonowosc miesieczna")
)

precip_heatmap

alt.Chart(...)

### 3.6 Wizualizacja 3: temperatura, wiatr i typ pogody

Wykres punktowy pozwala zobaczyc relacje miedzy zmiennymi ilosciowymi. Kolor pokazuje kategorie pogody, a tooltip daje szybki dostep do szczegolow dnia.

In [47]:
weather_scatter = (
    alt.Chart(weather_clean)
    .mark_circle(size=65, opacity=0.72)
    .encode(
        x=alt.X("temp_mid:Q", title="Srednia temperatura dzienna"),
        y=alt.Y("wind:Q", title="Wiatr"),
        color=alt.Color("weather:N", title="Pogoda"),
        size=alt.Size("precipitation:Q", title="Opady", scale=alt.Scale(range=[20, 220])),
        tooltip=[
            alt.Tooltip("date:T", title="Data"),
            alt.Tooltip("weather:N", title="Pogoda"),
            alt.Tooltip("temp_mid:Q", title="Srednia temp.", format=".1f"),
            alt.Tooltip("wind:Q", title="Wiatr", format=".1f"),
            alt.Tooltip("precipitation:Q", title="Opady", format=".1f"),
        ],
    )
    .properties(width=620, height=360, title="Temperatura, wiatr i opady w danych dziennych")
)

weather_scatter

alt.Chart(...)

### 3.7 Element interaktywny: selection + linked views

Altair pozwala opisac interakcje deklaratywnie. W aktualnym API Altaira parametry i selekcje dodajemy przez `add_params()`.

Ponizej:

- przeciagamy zakres na wykresie punktowym,
- punkty poza zaznaczeniem bledna,
- histogram opadow filtruje sie do zaznaczonych obserwacji.


In [49]:
brush = alt.selection_interval(encodings=["x", "y"])
points = (
    alt.Chart(weather_clean)
    .mark_circle(size=60)
    .encode(
        x=alt.X("temp_mid:Q", title="Srednia temperatura dzienna"),
        y=alt.Y("wind:Q", title="Wiatr"),
        color=alt.condition(brush, "weather:N", alt.value("#d6d6d6")),
        opacity=alt.condition(brush, alt.value(0.85), alt.value(0.22)),
        tooltip=["date:T", "weather:N", "temp_mid:Q", "wind:Q", "precipitation:Q"],
    )
    .add_params(brush)
    .properties(width=450, height=320, title="Zaznacz zakres temperatury i wiatru")
)

bars = (
    alt.Chart(weather_clean)
    .mark_bar()
    .encode(
        x=alt.X("weather:N", title="Typ pogody"),
        y=alt.Y("count():Q", title="Liczba dni"),
        color=alt.Color("weather:N", legend=None),
        tooltip=[alt.Tooltip("count():Q", title="Liczba dni")],
    )
    .transform_filter(brush)
    .properties(width=280, height=320, title="Rozklad w zaznaczeniu")
)

points | bars

alt.HConcatChart(...)

### 3.7 Drugi typ interakcji: kontrolka UI

Parametr moze byc powiazany z kontrolka HTML, np. lista rozwijana. Tutaj wybieramy rok i aktualizujemy wykres bez pisania JavaScriptu.

In [50]:
year_options = sorted(weather_clean["year"].unique().tolist())

year_param = alt.param(
    name="Rok",
    value=year_options[0],
    bind=alt.binding_select(options=year_options, name="Wybierz rok: "),
)

daily_for_year = (
    alt.Chart(weather_clean)
    .mark_bar()
    .encode(
        x=alt.X("monthdate(date):T", title="Dzien roku"),
        y=alt.Y("precipitation:Q", title="Opady"),
        color=alt.Color("weather:N", title="Pogoda"),
        tooltip=["date:T", "weather:N", "precipitation:Q", "temp_min:Q", "temp_max:Q"],
    )
    .add_params(year_param)
    .transform_filter(alt.datum.year == year_param)
    .properties(width=760, height=260, title="Dzienne opady w wybranym roku")
)

daily_for_year

alt.Chart(...)

## 4. Funkcje zaawansowane (5-8 min)

### Optymalizacja pod duze dane

Altair domyslnie osadza dane w specyfikacji wykresu. To jest wygodne, ale przy duzych tabelach moze prowadzic do bardzo duzych notebookow i wolniejszego renderowania. Dokumentacja opisuje limit `MaxRowsError` dla danych osadzanych bezposrednio i rekomenduje przemyslenie sposobu przekazywania danych.

Typowe strategie:

- agreguj dane przed wykresem, jesli wykres pokazuje trendy lub rozklady,
- przekazuj dane jako URL lub plik zamiast osadzac wszystko w notebooku,
- wlacz `vegafusion`, gdy uzywasz transformacji Vega-Lite i chcesz przeliczac je efektywniej po stronie Pythona.

### Customizacja

Customizacja w Altairze zwykle dotyczy:

- `scale`: zakresy, palety kolorow, domeny,
- `axis`: format osi, etykiety, tytuly,
- `legend`: polozenie i tytul legendy,
- `properties`: szerokosc, wysokosc, tytul,
- `configure_*`: globalny styl wykresu.

### Wydajnosc

Najwiekszy wplyw na wydajnosc ma liczba punktow wysylanych do przegladarki i zlozonosc interakcji. W praktyce czesto lepiej pokazac agregat lub histogram niz setki tysiecy surowych punktow.

Normalnie przepływ jest taki:
```
Altair -> Vega-Lite JSON -> przeglądarka -> Vega JS robi transformacje i rysuje
```

VegaFusion zmienia to tak:
```
Altair -> Vega-Lite JSON -> VegaFusion wykonuje transformacje w Pythonie/Rust -> przeglądarka dostaje mniejsze, gotowe dane -> Vega JS tylko renderuje

```

In [ ]:
!pip install vegafusion

In [51]:
# Opcjonalnie dla wiekszych danych, po instalacji: !pip install vegafusion
alt.data_transformers.enable("vegafusion")

# Alternatywy:
# alt.data_transformers.disable_max_rows()  # uzywac ostroznie
# weather_sample = weather_clean.sample(500, random_state=42)
# monthly zamiast weather_clean, jesli wystarcza agregacja miesieczna

DataTransformerRegistry.enable('vegafusion')

In [52]:
custom_chart = (
    alt.Chart(monthly)
    .mark_area(opacity=0.75, line=True)
    .encode(
        x=alt.X("yearmonth:T", title="Miesiac"),
        y=alt.Y("avg_temp:Q", title="Srednia temperatura", stack=None),
        color=alt.Color("year:N", title="Rok", scale=alt.Scale(scheme="tableau10")),
        tooltip=["year:O", "month_name:N", alt.Tooltip("avg_temp:Q", format=".1f")],
    )
    .transform_calculate(yearmonth="datetime(2000, datum.month - 1, 1)")
    .properties(width=760, height=300, title="Porownanie sezonowosci temperatury miedzy latami")
    .configure_view(stroke=None)
    .configure_axis(labelFontSize=12, titleFontSize=13)
    .configure_title(fontSize=16, anchor="start")
)

custom_chart

alt.Chart(...)

### Czy da sie zrobic plot 3D w Altairze?

Krotka odpowiedz: **nie w takim sensie jak w Plotly, Mayavi albo PyVista**. Altair jest nakladka na Vega-Lite, a Vega-Lite jest przede wszystkim biblioteka do **dwuwymiarowych, deklaratywnych wizualizacji statystycznych**.

Co mozna zrobic zamiast 3D:

- zakodowac trzecia zmienna kolorem, rozmiarem albo przezroczystoscia punktu,
- zrobic facet/grid malych wykresow dla kolejnych kategorii,
- dodac interaktywne filtrowanie zamiast obracania sceny 3D,
- uzyc heatmapy albo contour-like podejscia, jezeli dane tworza powierzchnie.

Kiedy naprawde potrzebujemy obracanej sceny 3D, lepiej uzyc `plotly`, `pyvista`, `mayavi` albo `three.js`. Do analizy danych 2D Altair czesto jest czytelniejszy, bo nie ukrywa wartosci przez perspektywe i zaslanianie punktow.

### Piekny plot z mapa: choropleth

Altair potrafi robic mapy przez `mark_geoshape()`. Najczestszy schemat to:

- geometria mapy, np. granice stanow, krajow albo hrabstw,
- tabela z wartosciami liczbowymi,
- `transform_lookup()`, ktory laczy geometrie z tabela po wspolnym kluczu,
- kolor jako zakodowana wartosc.

Wazny detal: klucze musza pasowac. Zbior `data.unemployment` jest po hrabstwach USA, wiec laczymy go z geometria `counties`, a nie `states`. Przed mapa wracamy tez do domyslnego data transformera, bo `vegafusion` jest swietny do duzych tabel, ale przy topojson/geo czasem niepotrzebnie komplikuje renderowanie w notebooku.


In [53]:
# Przy mapach topojson najbezpieczniej wrocic do domyslnego transformera.
alt.data_transformers.enable("default")

counties = alt.topo_feature(data.us_10m.url, feature="counties")
unemployment = data.unemployment.url

map_chart = (
    alt.Chart(counties)
    .mark_geoshape(stroke="#ffffff", strokeWidth=0.15)
    .encode(
        color=alt.Color(
            "rate_percent:Q",
            title="Bezrobocie (%)",
            scale=alt.Scale(scheme="viridis"),
        ),
        tooltip=[
            alt.Tooltip("id:O", title="ID hrabstwa"),
            alt.Tooltip("rate_percent:Q", title="Stopa bezrobocia (%)", format=".1f"),
        ],
    )
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(unemployment, key="id", fields=["rate"]),
    )
    .transform_calculate(rate_percent="datum.rate * 100")
    .project(type="albersUsa")
    .properties(
        width=760,
        height=450,
        title="Mapa choropleth: bezrobocie w hrabstwach USA",
        background="#f7f9fb",
    )
    .configure_title(fontSize=18, anchor="start", color="#1f2937")
    .configure_legend(labelFontSize=12, titleFontSize=13)
    .configure_view(stroke=None)
)

map_chart


alt.Chart(...)

## 5. Mini-zadanie dla grupy (5-8 min)

### Challenge

Zrob wykres pokazujacy: **w ktorych miesiacach Seattle ma najwiecej dni deszczowych i jak to rozni sie miedzy latami**.

Wymagania:

- uzyj danych `monthly`,
- os X: miesiac,
- os Y: liczba dni deszczowych,
- kolor: rok albo typ agregacji,
- dodaj tooltip,
- dodaj przynajmniej jeden element interaktywny: zoom, tooltip, selekcje albo dropdown.

Czas pracy: 4-5 minut

In [54]:
monthly.head()

,year,month,month_name,avg_temp,total_precipitation,rainy_days,avg_wind
0,2012,1,January,4.298387,173.3,22,3.900000
1,2012,2,February,6.239655,92.3,19,3.903448
2,2012,3,March,6.196774,183.0,22,4.248387
3,2012,4,April,10.433333,68.1,17,3.373333
4,2012,5,May,12.925806,52.2,10,3.354839


In [ ]:
# Miejsce na rozwiazanie mini-zadania.
# Podpowiedz: zacznij od alt.Chart(monthly).mark_line().encode(...)

highlight_year = alt.selection_point(fields=["year"], bind="legend")


# TODO
# rainy_days_solution = (
#     alt.Chart(monthly)
#     .mark_line()
#     .encode()
# )


# rainy_days_solution

alt.Chart(...)

## 6. Dyskusja

Pytania do grupy:

- Kiedy wybralibyscie Altaira zamiast Matplotliba albo Seaborna?
- Czy deklaratywny sposob myslenia jest bardziej czytelny, czy na poczatku trudniejszy?
- Co zmienilibyscie, gdyby dane mialy 10 milionow wierszy?
